In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

DATA_PATH = Path('data') / 'csv'

In [7]:
# === Cell 2: load cleaned data from EDA notebook ===
transactions_df = pd.read_csv(DATA_PATH / 'transactions.csv', parse_dates=['WEEK_END_DATE'])
products_df     = pd.read_csv(DATA_PATH / 'products.csv')
stores_df       = pd.read_csv(DATA_PATH / 'stores.csv')

print('transactions:', transactions_df.shape)
print('products    :', products_df.shape)
print('stores      :', stores_df.shape)

transactions: (524950, 12)
products    : (58, 6)
stores      : (79, 9)


In [8]:
mask_clean = (
    transactions_df['PRICE'].notna() &
    transactions_df['BASE_PRICE'].notna() &
    (transactions_df['PRICE'] > 0) &
    (transactions_df['BASE_PRICE'] > 0) &
    (transactions_df['UNITS'] > 0)
)
df = transactions_df.loc[mask_clean].copy()

cereal_upcs = products_df.loc[products_df['CATEGORY'] == 'COLD CEREAL', 'UPC'].tolist()
df = df[df['UPC'].isin(cereal_upcs)].copy()

stores_clean = stores_df.drop_duplicates(subset='STORE_ID', keep='first').copy()

df = df.merge(
    products_df[['UPC', 'DESCRIPTION', 'MANUFACTURER', 'SUB_CATEGORY', 'PRODUCT_SIZE']],
    on='UPC', how='left', validate='many_to_one'
)
df = df.merge(
    stores_clean[['STORE_ID', 'SEG_VALUE_NAME', 'MSA_CODE', 'SALES_AREA_SIZE_NUM', 'AVG_WEEKLY_BASKETS']]
        .rename(columns={'STORE_ID': 'STORE_NUM'}),
    on='STORE_NUM', how='left', validate='many_to_one'
)

assert df['SEG_VALUE_NAME'].isna().sum() == 0
print('After merges:', df.shape)

After merges: (169677, 20)


In [9]:
# 1) Log-transformed target и treatment
df['log_units']      = np.log(df['UNITS'])
df['log_price']      = np.log(df['PRICE'])
df['log_base_price'] = np.log(df['BASE_PRICE'])

# 2) Глубина скидки
df['discount_depth'] = (df['BASE_PRICE'] - df['PRICE']) / df['BASE_PRICE']
df['on_promo']       = (df['discount_depth'] > 0.01).astype(int)

# 3) Промо-механика в одной переменной
def promo_type(row):
    if row['FEATURE'] == 1 and row['DISPLAY'] == 1:
        return 'feature_display'
    if row['FEATURE'] == 1:
        return 'feature'
    if row['DISPLAY'] == 1:
        return 'display'
    if row['TPR_ONLY'] == 1:
        return 'tpr'
    return 'none'

df['promo_type'] = df.apply(promo_type, axis=1)

# 4) Временные признаки
df['year']         = df['WEEK_END_DATE'].dt.year
df['month']        = df['WEEK_END_DATE'].dt.month
df['week_of_year'] = df['WEEK_END_DATE'].dt.isocalendar().week.astype(int)
df['quarter']      = df['WEEK_END_DATE'].dt.quarter

holidays_weeks = pd.to_datetime([
    '2009-01-04', '2009-02-15', '2009-05-31', '2009-07-05', '2009-09-06',
    '2009-11-29', '2009-12-27',
    '2010-01-03', '2010-02-14', '2010-05-30', '2010-07-04', '2010-09-05',
    '2010-11-28', '2010-12-26',
    '2011-01-02', '2011-02-13', '2011-05-29', '2011-07-03', '2011-09-04',
    '2011-11-27', '2011-12-25',
])
df['is_holiday_week'] = df['WEEK_END_DATE'].isin(holidays_weeks).astype(int)

# 5) Размер упаковки
df['size_oz']      = df['PRODUCT_SIZE'].str.replace(' OZ', '', regex=False).astype(float)
df['price_per_oz'] = df['PRICE'] / df['size_oz']

# 6) Категориальные коды
df['upc_code']    = df['UPC'].astype('category').cat.codes
df['store_code'] = df['STORE_NUM'].astype('category').cat.codes
df['manuf_code']  = df['MANUFACTURER'].astype('category').cat.codes
df['subcat_code'] = df['SUB_CATEGORY'].astype('category').cat.codes
df['seg_code']    = df['SEG_VALUE_NAME'].astype('category').cat.codes
df['promo_code']  = df['promo_type'].astype('category').cat.codes

df = df.sort_values(['STORE_NUM', 'UPC', 'WEEK_END_DATE']).reset_index(drop=True)

print('Final shape:', df.shape)
print('\npromo_type distribution:')
print(df['promo_type'].value_counts(normalize=True).round(3))
print('\nKey numeric features:')
print(df[['log_units', 'log_price', 'discount_depth', 'on_promo',
         'size_oz', 'price_per_oz', 'is_holiday_week']].describe().round(3))

Final shape: (169677, 39)

promo_type distribution:
promo_type
none               0.763
tpr                0.110
feature_display    0.056
feature            0.039
display            0.032
Name: proportion, dtype: float64

Key numeric features:
        log_units   log_price  discount_depth    on_promo     size_oz  price_per_oz  is_holiday_week
count  169677.000  169677.000      169677.000  169677.000  169677.000    169677.000         169677.0
mean        3.135       0.974           0.050       0.226      15.021         0.188              0.0
std         0.901       0.263           0.106       0.419       2.921         0.051              0.0
min         0.000      -0.545          -1.397       0.000      11.000         0.046              0.0
25%         2.639       0.784           0.000       0.000      12.250         0.152              0.0
50%         3.178       1.001           0.000       0.000      14.000         0.187              0.0
75%         3.689       1.160           0.000    

In [16]:
import holidays

us_holidays_2009_2012 = holidays.US(years=[2009, 2010, 2011, 2012])

big_holidays = {
    "New Year's Day",
    "Memorial Day",
    "Independence Day",
    "Labor Day",
    "Thanksgiving Day",
    "Christmas Day",
}

holiday_dates = pd.to_datetime([
    date for date, name in us_holidays_2009_2012.items()
    if name in big_holidays
])

all_weeks = pd.Series(sorted(df['WEEK_END_DATE'].unique()))

def retail_week_end(date):
    """Конец розничной недели (среда), в которую попадает date.
    Это первая среда из датасета, которая >= date."""
    after = all_weeks[all_weeks >= date]
    return after.iloc[0] if len(after) > 0 else pd.NaT

holiday_weeks = pd.Series([retail_week_end(d) for d in holiday_dates]).dropna().unique()
df['is_holiday_week'] = df['WEEK_END_DATE'].isin(holiday_weeks).astype(int)

# Проверка
print(f'Distinct holiday weeks: {len(holiday_weeks)}')
print(f'Share of holiday-week rows: {df["is_holiday_week"].mean():.3f}')

# По годам
print('\nBy year:')
holiday_weeks_df = pd.DataFrame({'week_end': sorted(holiday_weeks)})
holiday_weeks_df['year'] = holiday_weeks_df['week_end'].dt.year
print(holiday_weeks_df.groupby('year').size())

print('\nAll holiday weeks:')
for w in sorted(holiday_weeks):
    print(' ', w.date(), '—', w.day_name())

Distinct holiday weeks: 19
Share of holiday-week rows: 0.122

By year:
year
2009    6
2010    6
2011    6
2012    1
dtype: int64

All holiday weeks:
  2009-01-14 — Wednesday
  2009-05-27 — Wednesday
  2009-07-08 — Wednesday
  2009-09-09 — Wednesday
  2009-12-02 — Wednesday
  2009-12-30 — Wednesday
  2010-01-06 — Wednesday
  2010-06-02 — Wednesday
  2010-07-07 — Wednesday
  2010-09-08 — Wednesday
  2010-12-01 — Wednesday
  2010-12-29 — Wednesday
  2011-01-05 — Wednesday
  2011-06-01 — Wednesday
  2011-07-06 — Wednesday
  2011-09-07 — Wednesday
  2011-11-30 — Wednesday
  2011-12-28 — Wednesday
  2012-01-04 — Wednesday


In [17]:
# масштаб выбросов PRICE > BASE_PRICE
above_base = df[df['PRICE'] > df['BASE_PRICE']].copy()
above_base['markup'] = (above_base['PRICE'] - above_base['BASE_PRICE']) / above_base['BASE_PRICE']

print(f'Rows with PRICE > BASE_PRICE: {len(above_base)} ({len(above_base)/len(df)*100:.2f}%)')
print('\nmarkup describe:')
print(above_base['markup'].describe(percentiles=[.5, .9, .95, .99]).round(3))

cols = ['WEEK_END_DATE', 'STORE_NUM', 'UPC', 'DESCRIPTION', 'PRICE', 'BASE_PRICE', 'markup', 'UNITS']
print('\nWorst 10 cases:')
print(above_base.nlargest(10, 'markup')[cols].to_string(index=False))

Rows with PRICE > BASE_PRICE: 678 (0.40%)

markup describe:
count    678.000
mean       0.043
std        0.073
min        0.003
50%        0.025
90%        0.093
95%        0.133
99%        0.295
max        1.397
Name: markup, dtype: float64

Worst 10 cases:
WEEK_END_DATE  STORE_NUM         UPC               DESCRIPTION  PRICE  BASE_PRICE   markup  UNITS
   2009-06-17      17599  3000006340        QKER LIFE ORIGINAL   7.00        2.92 1.397260      8
   2010-08-25      26981  3000006340        QKER LIFE ORIGINAL   4.09        2.74 0.492701      5
   2010-08-18      26981  3000006340        QKER LIFE ORIGINAL   3.90        2.74 0.423358     11
   2010-09-01        387  3000006340        QKER LIFE ORIGINAL   4.09        2.97 0.377104     11
   2010-08-25      26973  3000006340        QKER LIFE ORIGINAL   4.09        2.97 0.377104      4
   2010-08-25        387  3000006340        QKER LIFE ORIGINAL   3.92        2.97 0.319865     12
   2010-10-06      21213  3000006340        QKER LIFE O

In [21]:
# финализация фич и сохранение

# 1) Считаем markup явно и режем экстремум
df['markup'] = ((df['PRICE'] - df['BASE_PRICE']) / df['BASE_PRICE']).clip(lower=0)

# Выкинем единичные экстремальные markup'ы (> 50%) — явный шум
mask_extreme = df['markup'] > 0.5
print(f'Dropping extreme markup rows: {mask_extreme.sum()}')
df = df[~mask_extreme].copy()

# 2) discount_depth теперь чистый: >0 если есть скидка, иначе 0
df['discount_depth'] = ((df['BASE_PRICE'] - df['PRICE']) / df['BASE_PRICE']).clip(lower=0)
df['on_promo']       = (df['discount_depth'] > 0.01).astype(int)

# Проверка
print('\ndiscount_depth describe (after fix):')
print(df['discount_depth'].describe(percentiles=[.5, .9, .95, .99]).round(3))

print('\nFinal shape:', df.shape)
print('Columns:', len(df.columns))

# 3) Сохраняем
out_path = Path('data') / 'features'
out_path.mkdir(exist_ok=True)
out_path = Path('data') / 'features' / 'cereal_features.parquet'
df.to_parquet(out_path, index=False)
print(f'\nSaved {df.shape} → {out_path}')
print(f'File size: {out_path.stat().st_size / 1024:.1f} KB')

# Финальный обзор фич
print('\n=== Feature summary ===')
feature_groups = {
    'target & treatment': ['UNITS', 'log_units', 'PRICE', 'log_price', 'BASE_PRICE'],
    'discount/promo'    : ['discount_depth', 'on_promo', 'promo_type', 'markup'],
    'time'              : ['WEEK_END_DATE', 'year', 'month', 'week_of_year', 'quarter', 'is_holiday_week'],
    'product'           : ['UPC', 'DESCRIPTION', 'MANUFACTURER', 'SUB_CATEGORY', 'size_oz', 'price_per_oz'],
    'store'             : ['STORE_NUM', 'SEG_VALUE_NAME', 'MSA_CODE', 'SALES_AREA_SIZE_NUM'],
    'codes for ML'      : ['upc_code', 'store_code', 'manuf_code', 'subcat_code', 'seg_code', 'promo_code'],
}
for group, cols in feature_groups.items():
    present = [c for c in cols if c in df.columns]
    print(f'{group:>22}: {present}')

Dropping extreme markup rows: 0

discount_depth describe (after fix):
count    169676.000
mean          0.050
std           0.106
min           0.000
50%           0.000
90%           0.216
95%           0.303
99%           0.445
max           0.824
Name: discount_depth, dtype: float64

Final shape: (169676, 40)
Columns: 40

Saved (169676, 40) → data/features/cereal_features.parquet
File size: 2321.2 KB

=== Feature summary ===
    target & treatment: ['UNITS', 'log_units', 'PRICE', 'log_price', 'BASE_PRICE']
        discount/promo: ['discount_depth', 'on_promo', 'promo_type', 'markup']
                  time: ['WEEK_END_DATE', 'year', 'month', 'week_of_year', 'quarter', 'is_holiday_week']
               product: ['UPC', 'DESCRIPTION', 'MANUFACTURER', 'SUB_CATEGORY', 'size_oz', 'price_per_oz']
                 store: ['STORE_NUM', 'SEG_VALUE_NAME', 'MSA_CODE', 'SALES_AREA_SIZE_NUM']
          codes for ML: ['upc_code', 'store_code', 'manuf_code', 'subcat_code', 'seg_code', 'promo_code'